In [ ]:
import wave, os, glob
import noisereduce
from scipy.io import wavfile
import matplotlib.pyplot as plt
import librosa.display
import sys
import librosa
import numpy as np
np.set_printoptions(threshold=sys.maxsize)
import math
from scipy.stats import variation 
import statistics
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Project directory structure
#
# ATAS_CWNS_CWS_Project/        ← THIS is base_dir
# │
# ├── Audio_files_clipped/      ← create this folder; clipped audio files stored here
# ├── Visualize/                ← create this folder; .wav audio files for visualization/analysis stored here
# ├── Stat_csv_files/           ← input CSV files and summary statistics CSV files stored here
# ├── Individual_OutputCSV_Files/ ← individual output CSV files stored here
# │
# ├── Notebooks/                ← notebooks stored here
# │   ├── ATAS_Visualization/   ← visualization notebook
# │   └── ML_Analysis/          ← machine learning analysis notebooks
# │
# └── Scripts/                  ← function notebooks
#
base_dir = '..../ATAS_CWNS_CWS_Project'  # Change to required project directory

# Load the all details csv 
par_details = pd.read_csv(base_dir + "/Stat_csv_files/CWNS_CWS_all_details.csv") # Change to required destination

In [ ]:
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping

# ============================================================
# LSTM: LIGHT nested CV with saved foldwise metrics + summary
# Outputs:
#  - lstm_foldwise.csv
#  - lstm_summary.csv (mean, sd, min, max, n for acc/f1/precision/recall)
#  - figures: fold metric lines + boxplots + hyperparam counts
# ============================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

required_cols = [
    "All_events_durations", "Event_type", "Age",
    "Pause_Duration_s", "Pause_Events", "Mean Speech_s",
    "long_p_count", "short_p_count", "Speech_Rate", "Group",
]
missing = [c for c in required_cols if c not in par_details.columns]
if missing:
    raise ValueError(f"par_details is missing required columns: {missing}")

def parse_array_cell(x):
    if isinstance(x, (list, tuple, np.ndarray)):
        return np.asarray(x, dtype=float)
    if pd.isna(x):
        return np.array([], dtype=float)
    return np.fromstring(str(x), sep=",")

df_raw = par_details.copy(deep=True)
for c in ["All_events_durations", "Event_type"]:
    df_raw[c] = df_raw[c].apply(parse_array_cell)

df_all = df_raw[
    [
        "All_events_durations", "Event_type", "Age",
        "Pause_Duration_s", "Pause_Events", "Mean Speech_s",
        "long_p_count", "short_p_count", "Speech_Rate", "Group"
    ]
].copy()

label_map = {"CWS": 0, "CWNS": 1}
if not set(df_all["Group"].unique()).issubset(set(label_map.keys())):
    raise ValueError(f"Unexpected Group labels found: {df_all['Group'].unique().tolist()}")

df_all["Numerical_Label"] = df_all["Group"].map(label_map).astype(np.int32)

# Participant-level scaling (global)
feature_cols = ["Age", "Pause_Duration_s", "Pause_Events", "Mean Speech_s", "long_p_count", "short_p_count", "Speech_Rate"]
feat_scaler = StandardScaler()
df_all[feature_cols] = feat_scaler.fit_transform(df_all[feature_cols])

# Build per-participant event frames
data_frames_cwns, data_frames_cws = [], []

for _, row in df_all.iterrows():
    durations = row["All_events_durations"]
    etypes = row["Event_type"]

    if len(durations) == 0 or len(etypes) == 0:
        continue

    if len(durations) != len(etypes):
        m = min(len(durations), len(etypes))
        durations = durations[:m]
        etypes = etypes[:m]

    seg = pd.DataFrame({"Time_diff": durations, "Event_type": etypes})

    # Normalize Time_diff within participant
    time_scaler = StandardScaler()
    seg["Time_diff"] = time_scaler.fit_transform(seg[["Time_diff"]])

    base = row[feature_cols].to_frame().T
    n = len(seg)
    repeated = pd.DataFrame(np.repeat(base.to_numpy(), n, axis=0), columns=base.columns)

    concatenated = pd.concat([seg.reset_index(drop=True), repeated.reset_index(drop=True)], axis=1)

    if row["Numerical_Label"] == 1:
        data_frames_cwns.append(concatenated)
    else:
        data_frames_cws.append(concatenated)

if len(data_frames_cwns) < 3 or len(data_frames_cws) < 3:
    raise ValueError(f"Not enough participants after preprocessing. CWNS={len(data_frames_cwns)}, CWS={len(data_frames_cws)}")

no_of_features = len(data_frames_cwns[0].columns)
print("no_of_features:", no_of_features)

def segment_dataframe(df, sample_length):
    segments = []
    num_full = len(df) // sample_length

    for i in range(num_full):
        start = i * sample_length
        end = start + sample_length
        segments.append(df.iloc[start:end, :])

    rem = len(df) % sample_length
    if rem != 0:
        tail = df.iloc[num_full * sample_length :, :]
        padding = pd.DataFrame(0, index=range(sample_length - len(tail)), columns=df.columns)
        segments.append(pd.concat([tail, padding], axis=0))

    return segments

sample_length = 20
segments_cwns = [segment_dataframe(dfi, sample_length) for dfi in data_frames_cwns]
segments_cws  = [segment_dataframe(dfi, sample_length) for dfi in data_frames_cws]

# Precompute participant arrays
X_part, y_part = [], []

for plist in segments_cwns:
    arrs = [seg.to_numpy(dtype=np.float32, copy=False) for seg in plist]
    X_part.append(np.stack(arrs, axis=0))
    y_part.append(1)

for plist in segments_cws:
    arrs = [seg.to_numpy(dtype=np.float32, copy=False) for seg in plist]
    X_part.append(np.stack(arrs, axis=0))
    y_part.append(0)

y_part = np.array(y_part, dtype=int)
pids = np.arange(len(X_part))

print("Participants total:", len(pids), "| CWNS:", int(np.sum(y_part == 1)), "| CWS:", int(np.sum(y_part == 0)))

def build_segment_level_Xy(participant_indices):
    X = np.concatenate([X_part[i] for i in participant_indices], axis=0)
    y = np.concatenate([np.full((X_part[i].shape[0],), y_part[i]) for i in participant_indices], axis=0)
    return X.astype(np.float32, copy=False), y.astype(np.float32, copy=False)

def build_model(no_of_features, units=32, dropout=0.5):
    tf.keras.backend.clear_session()
    model = Sequential()
    model.add(LSTM(units=units, dropout=0.2, recurrent_dropout=0.2, input_shape=(None, no_of_features)))
    model.add(Dropout(dropout))
    model.add(Dense(1, activation="sigmoid"))
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

outer_cv = RepeatedStratifiedKFold(n_splits=3, n_repeats=2, random_state=SEED)
inner_cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=SEED)

param_grid = [
    {"units": 16, "dropout": 0.3},
    {"units": 32, "dropout": 0.5},
]

EPOCHS_INNER = 15
EPOCHS_FINAL = 25
BATCH_SIZE = 32
PATIENCE = 3

outer_metrics = []

for outer_fold, (outer_train_pids, outer_test_pids) in enumerate(outer_cv.split(pids, y_part), start=1):
    best_params = None
    best_score = -np.inf

    for params in param_grid:
        inner_scores = []

        for inner_train_idx, inner_val_idx in inner_cv.split(outer_train_pids, y_part[outer_train_pids]):
            inner_train_pids = outer_train_pids[inner_train_idx]
            inner_val_pids   = outer_train_pids[inner_val_idx]

            X_tr, y_tr = build_segment_level_Xy(inner_train_pids)
            X_va, y_va = build_segment_level_Xy(inner_val_pids)

            model = build_model(no_of_features, units=params["units"], dropout=params["dropout"])
            es = EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True)

            model.fit(
                X_tr, y_tr,
                validation_data=(X_va, y_va),
                epochs=EPOCHS_INNER,
                batch_size=BATCH_SIZE,
                verbose=0,
                callbacks=[es]
            )

            y_hat = (model.predict(X_va, verbose=0).ravel() >= 0.5).astype(int)
            inner_scores.append(f1_score(y_va.astype(int), y_hat))

        mean_inner = float(np.mean(inner_scores))
        if mean_inner > best_score:
            best_score = mean_inner
            best_params = params

    X_train, y_train = build_segment_level_Xy(outer_train_pids)
    X_test,  y_test  = build_segment_level_Xy(outer_test_pids)

    final_model = build_model(no_of_features, units=best_params["units"], dropout=best_params["dropout"])
    es = EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True)

    final_model.fit(
        X_train, y_train,
        validation_split=0.2,
        epochs=EPOCHS_FINAL,
        batch_size=BATCH_SIZE,
        verbose=0,
        callbacks=[es]
    )
           # --- Get segment-level probabilities ---
    y_prob = final_model.predict(X_test, verbose=0).ravel()

    # --- Aggregate to participant-level predictions ---
    participant_preds = []
    participant_true  = []

    idx = 0

    for pid in outer_test_pids:
        n_segments = X_part[pid].shape[0]

        # slice this participant's segment predictions
        seg_probs = y_prob[idx: idx + n_segments]
        idx += n_segments

        # --- majority vote over segments ---
        seg_preds = (seg_probs >= 0.5).astype(int)
        final_pred = int(np.mean(seg_preds) >= 0.5)

        participant_preds.append(final_pred)
        participant_true.append(y_part[pid])

    # --- Convert to arrays ---
    participant_preds = np.array(participant_preds)
    participant_true  = np.array(participant_true)

    # --- Compute participant-level metrics ---
    acc = accuracy_score(participant_true, participant_preds)
    f1  = f1_score(participant_true, participant_preds)
    prec, rec, _, _ = precision_recall_fscore_support(
        participant_true, participant_preds, average="binary"
    )
    outer_metrics.append({
        "Model": "LSTM",
        "Outer_Fold": int(outer_fold),
        "Accuracy": float(acc),
        "F1": float(f1),
        "Precision": float(prec),
        "Recall": float(rec),
        "Best_Params": str(best_params),
        "Best_Inner_F1": float(best_score),
        "n_train_segments": int(X_train.shape[0]),
        "n_test_segments": int(X_test.shape[0]),
    })

    print(
        f"[Outer fold {outer_fold:02d}] "
        f"ACC={acc:.3f} F1={f1:.3f} "
        f"Prec={prec:.3f} Rec={rec:.3f} "
        f"(best={best_params}, innerF1={best_score:.3f})"
    )

df_fold = pd.DataFrame(outer_metrics).sort_values("Outer_Fold")

# Summary with true ranges
summary = df_fold[["Accuracy", "F1", "Precision", "Recall"]].agg(["mean", "std", "min", "max", "count"])
print("\n=== LSTM LIGHT NESTED CV SUMMARY (mean, SD, min, max, n) ===")
print(summary)

# Save
df_fold.to_csv("lstm_foldwise.csv", index=False)
summary.reset_index().to_csv("lstm_summary.csv", index=False)

# Figures (paper-friendly)
plt.figure()
for col in ["Accuracy", "F1", "Precision", "Recall"]:
    plt.plot(df_fold["Outer_Fold"], df_fold[col], marker="o", label=col)
plt.title("LSTM Performance Across Outer Folds")
plt.xlabel("Outer Fold")
plt.ylabel("Score")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig("lstm_outerfold_metrics.png", dpi=300)
plt.show()

plt.figure()
data = [df_fold["Accuracy"].values, df_fold["F1"].values, df_fold["Precision"].values, df_fold["Recall"].values]
plt.boxplot(data, labels=["Accuracy", "F1", "Precision", "Recall"])
plt.title("LSTM Metric Distributions Across Outer Folds")
plt.ylabel("Score")
plt.tight_layout()
plt.savefig("lstm_metric_boxplots.png", dpi=300)
plt.show()

# Hyperparameter selection counts
df_fold["best_units"] = df_fold["Best_Params"].str.extract(r"'units':\s*(\d+)").astype(int)
df_fold["best_dropout"] = df_fold["Best_Params"].str.extract(r"'dropout':\s*([0-9.]+)").astype(float)
counts = df_fold.groupby(["best_units", "best_dropout"]).size()

plt.figure()
counts.plot(kind="bar")
plt.title("LSTM Best Hyperparameter Selection Counts")
plt.xlabel("(Units, Dropout)")
plt.ylabel("Count")
plt.grid(axis="y")
plt.tight_layout()
plt.savefig("lstm_best_param_counts.png", dpi=300)
plt.show()
